In [ ]:
from langgraph.graph import StateGraph, END,START
from typing import TypedDict
import os
from langchain_google_genai  import ChatGoogleGenerativeAI
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv())
model = ChatGoogleGenerativeAI(
    model= "gemini-2.5-flash"
)

In [109]:
class ChainState(TypedDict):
    job_discription: str
    resume_summery: str
    cover_letter: str

In [110]:
workflow = StateGraph(ChainState)
workflow

In [111]:
def generate_resume_summery(state:ChainState) -> ChainState:
    prompt = f"""
    You're a resume assistant. Read the following job discription and summarize the key qualifications and experience the ideal candidate should have, phrased as if from the perspective of a strong applicant's resume summary.

    Job Discription:
    {state['job_discription']}"""

    Response = model.invoke(prompt)
    
    return {**state, 'resume_summery':Response.content}




In [112]:
def generate_cover_letter(state:ChainState) -> ChainState:
    prompt = f""""
    You're a cover letter writing assistant. Using the resume summary below, write a professional and personalized cover letter for the following job.

    Job Discription:
    {state['job_discription']}
    Resume Summery:
    {state['resume_summery']}"""


    Response = model.invoke(prompt)

    return {**state, 'cover_letter': Response.content}


In [113]:
workflow.add_node("generate_resume_summery", generate_resume_summery)
workflow.add_node("generate_cover_letter",generate_cover_letter)

In [114]:
workflow.add_edge("generate_resume_summery", "generate_cover_letter")

In [115]:
workflow.set_entry_point("generate_resume_summery")

workflow.set_finish_point("generate_cover_letter")


In [116]:
app = workflow.compile()

In [117]:
input_state = {
    "job_discription": "We are looking for a data scientist with experience in machine learning, NLP, and Python. Prior work with large datasets and experience deploying models into production is required."
}


In [118]:
result = app.invoke(input_state)

In [ ]:
result['resume_summery']

'Here\'s a resume summary from the perspective of a strong applicant:\n\n"Highly experienced Data Scientist with deep expertise in machine learning, natural language processing (NLP), and Python. Proven track record of leveraging large datasets to develop and successfully deploy scalable models into production environments."'